# 06 — Optimize and serve with TensorRT-LLM

TensorRT-LLM is an advanced NVIDIA GPU inference runtime. Depending on the release and workflow, it can serve a supported Hugging Face checkpoint through its LLM API or build a hardware-aware TensorRT engine first. This step is for deployment optimization, not fine-tuning.

Nothing in this notebook installs TensorRT-LLM, builds an engine, or starts a container. Commands contain explicit placeholders because image tags, model support, conversion scripts, quantization formats, and CLI syntax are release-sensitive.

## Conceptual workflow

1. Finish LoRA training in NeMo and retain the base-model revision and adapter checkpoint.
2. Export or merge to a TensorRT-LLM-supported checkpoint format. Verify text generation before optimization.
3. Select a TensorRT-LLM release/container that supports the Qwen architecture and target GPU.
4. Choose precision or quantization using calibration data when required.
5. Either serve through the current PyTorch/LLM API backend or convert and build a TensorRT engine.
6. Start `trtllm-serve`, run correctness tests, then benchmark latency, throughput, memory, and concurrency.

Engine artifacts are usually tied to model settings, TensorRT-LLM/TensorRT versions, and GPU architecture. Record all of them.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

TRTLLM_IMAGE = 'nvcr.io/nvidia/tensorrt-llm/release:<TAG>'
EXPORTED_MODEL = '/workspace/project/outputs/exported_hf'
ENGINE_DIR = '/workspace/project/outputs/trtllm_engine'
print('Container placeholder:', TRTLLM_IMAGE)
print('Exported model placeholder:', EXPORTED_MODEL)
print('Engine output placeholder:', ENGINE_DIR)

## Start a release container

After selecting a real tag from the current TensorRT-LLM installation guide, run from the project root on a Linux GPU host:

```bash
export TRTLLM_IMAGE='nvcr.io/nvidia/tensorrt-llm/release:<TAG>'
docker run --rm -it --gpus all --ipc=host \
  --ulimit memlock=-1 --ulimit stack=67108864 \
  -p 8000:8000 \
  -v "$PWD:/workspace/project" \
  "$TRTLLM_IMAGE"
```

The host needs a supported NVIDIA driver, Docker, NVIDIA Container Toolkit, enough GPU memory, and enough disk for the source checkpoint plus generated artifacts.

## Path A — Current LLM API / direct serving

Recent TensorRT-LLM releases can serve supported model paths through `trtllm-serve`. Inside the selected container, inspect help first and then adapt this placeholder:

```bash
trtllm-serve --help
trtllm-serve serve /workspace/project/outputs/exported_hf \
  --backend tensorrt \
  --host 0.0.0.0 \
  --port 8000 \
  --max_batch_size 8 \
  --max_seq_len 1024
```

Some releases use `trtllm-serve <MODEL>` without the `serve` subcommand or default to a PyTorch backend. Treat `--help` in the pinned container as authoritative. Start with conservative sequence and batch limits.

## Path B — Explicit checkpoint conversion and engine build

The exact Qwen conversion entry point changes between releases. Locate the Qwen example in the selected container, then adapt the following conceptual commands:

```bash
# Placeholder: convert the verified exported checkpoint to a TRT-LLM checkpoint.
python <QWEN_CONVERT_SCRIPT> \
  --model_dir /workspace/project/outputs/exported_hf \
  --output_dir /workspace/project/outputs/trtllm_checkpoint \
  --dtype bfloat16

# Build an engine sized for the intended workload.
trtllm-build \
  --checkpoint_dir /workspace/project/outputs/trtllm_checkpoint \
  --output_dir /workspace/project/outputs/trtllm_engine \
  --max_batch_size 8 \
  --max_input_len 768 \
  --max_seq_len 1024

# Serve the built engine using the CLI syntax shown by this release.
trtllm-serve serve /workspace/project/outputs/trtllm_engine \
  --tokenizer /workspace/project/outputs/exported_hf \
  --host 0.0.0.0 --port 8000
```

Do not assume that a build that succeeds is correct. Compare outputs against the pre-optimization model before benchmarking.

In [ ]:
CALL_TRTLLM_SERVER = False

if CALL_TRTLLM_SERVER:
    from openai import OpenAI
    client = OpenAI(base_url='http://127.0.0.1:8000/v1', api_key='not-required')
    models = client.models.list()
    served_model = models.data[0].id
    response = client.chat.completions.create(
        model=served_model,
        messages=[{'role': 'user', 'content': 'Define LoRA in one sentence.'}],
        temperature=0,
        max_tokens=96,
    )
    print(response.choices[0].message.content)
else:
    print('TensorRT-LLM server call skipped.')

## Acceptance checks

Before calling the optimization successful, run the Notebook 4 prompt set against the unoptimized and optimized endpoints, compare structured-output validity and task metrics, test maximum intended context, and inspect numerical regressions. Then benchmark warm and cold starts, p50/p95/p99 latency, time to first token, output-token throughput, peak GPU memory, concurrency, and failure behavior.

Notebook 7 moves from an optimization toolkit to NVIDIA NIM, a packaged microservice deployment option.